# How To Split Data Effectively for Your Data Science Project

Companion to the [complete article](https://todatabeyond.com/blog/how-to-split-data-effectively-for-your-data-science-project).

Run all cells in order with Python 3.10+. CPU only, no API keys. Setup installs the tested versions; the first example downloads the public California housing CSV from the original book repository (about 1.4 MB). Original ten gist techniques are retained. Direct CSV download avoids extracting an archive. CRC32 uses explicit little-endian integer bytes, and category assignment supports pandas Copy-on-Write.

A row-index ID only works for append-only records whose existing indices never change. The source geographic composite is illustrative: it has collisions in this dataset, so use a genuine immutable unique record ID for new projects. Hash splitting targets an approximate fraction, not an exact 80/20 count. Regression checks demonstrate these boundaries. No outputs or execution counts are committed.


In [ ]:
%pip install --quiet numpy==2.3.3 pandas==2.3.3 scikit-learn==1.7.2 matplotlib==3.10.9


## 1. Load California housing data


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import numpy as np
import pandas as pd

HOUSING_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
HOUSING_PATH = Path("datasets/housing")

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    housing_path.mkdir(parents=True, exist_ok=True)
    csv_path = housing_path / "housing.csv"
    urlretrieve(housing_url, csv_path)
    return csv_path

def load_housing_data(housing_path=HOUSING_PATH):
    return pd.read_csv(housing_path / "housing.csv")

csv_path = fetch_housing_data()
housing = load_housing_data()
print("Dataset shape:", housing.shape)
print("SHA-256:", hashlib.sha256(csv_path.read_bytes()).hexdigest())
print(housing.head().to_string(index=False))


## 2. Custom random split


In [ ]:
np.random.seed(42)

#define a splitting fucntion 
def split_train_test(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

# split the data into training and testing data 
train_set, test_set = split_train_test(housing, 0.2)

print('The size of the training data', len(train_set))
print('The size of the testing data', len(test_set))


## 3. Stable ID membership


In [ ]:
from zlib import crc32
def test_set_check(identifier, test_ratio):
    return crc32(int(identifier).to_bytes(8, byteorder="little", signed=True)) & 0xffffffff < test_ratio * 2**32

def split_train_test_by_id(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

print("Stable-ID split functions ready.")


## 4. Use row-index IDs


In [ ]:
housing_with_id = housing.reset_index() # adds an `index` column
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "index")

print("Train rows:", len(train_set), "Test rows:", len(test_set))


## 5. Use geographic IDs


In [ ]:
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "id")

print("Train rows:", len(train_set), "Test rows:", len(test_set))


## 6. Scikit-learn random split


In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

print("Train rows:", len(train_set), "Test rows:", len(test_set))


## 7. Plot income distribution


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

housing['median_income'].hist(bins=50, figsize=(20,15))
plt.xlabel('Median Income')
plt.ylabel('Count')
plt.title('Histogram for the median income')
plt.show()
print("Median income histogram: 50 bins;", len(housing), "districts.")


## 8. Create income categories


In [ ]:
housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)
housing["income_cat"] = housing["income_cat"].where(housing["income_cat"] < 5, 5.0)
housing["income_cat"].hist()
plt.xlabel('Median Income')
plt.ylabel('Count')
plt.title('Histogram for the median income')
plt.show()
print(housing["income_cat"].value_counts().sort_index().to_string())


## 9. Stratified sampling


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_index].copy()
    strat_test_set = housing.iloc[test_index].copy()

print("Stratified train rows:", len(strat_train_set))
print("Stratified test rows:", len(strat_test_set))


## 10. Test-set proportions


In [ ]:
print((strat_test_set["income_cat"].value_counts() / len(strat_test_set)).to_string())


## 11. Full-dataset proportions


In [ ]:
print((housing["income_cat"].value_counts() / len(housing)).to_string())


## Regression checks

Verify partitions, seeded reproducibility, append/reordering stability with immutable IDs, coordinate collisions and category proportions.


In [ ]:
# Every split is a complete, disjoint partition of the input rows.
def check_partition(data, train, test):
    assert len(train) + len(test) == len(data)
    assert set(train.index).isdisjoint(test.index)
    assert set(train.index) | set(test.index) == set(data.index)

np.random.seed(42)
first_train, first_test = split_train_test(housing, 0.2)
np.random.seed(42)
again_train, again_test = split_train_test(housing, 0.2)
assert first_test.index.equals(again_test.index)
check_partition(housing, first_train, first_test)
assert (len(first_train), len(first_test)) == (16512, 4128)

# A fresh seeded permutation does not preserve membership after appending rows.
old = pd.DataFrame({"record_id": np.arange(1000)})
new = pd.DataFrame({"record_id": np.arange(1200)})
np.random.seed(42)
_, old_random = split_train_test(old, 0.2)
np.random.seed(42)
_, new_random = split_train_test(new, 0.2)
assert set(old_random.record_id) != (set(new_random.record_id) & set(old.record_id))

# Hash membership stays stable across both appends and row reordering.
old_train, old_test = split_train_test_by_id(old, 0.2, "record_id")
new_train, new_test = split_train_test_by_id(new, 0.2, "record_id")
_, reordered_test = split_train_test_by_id(new.sample(frac=1, random_state=7), 0.2, "record_id")
check_partition(new, new_train, new_test)
assert set(old_test.record_id) == (set(new_test.record_id) & set(old.record_id))
assert set(new_test.record_id) == set(reordered_test.record_id)
assert not set(old_train.record_id) & set(new_test.record_id)
assert not test_set_check(123, 0.0) and test_set_check(123, 1.0)

# Keep the source's coordinate example, but do not assume it is a unique key.
coordinate_keys = housing_with_id["id"].map(int)
duplicate_rows = coordinate_keys.duplicated().sum()
assert duplicate_rows > 0
coordinate_membership = coordinate_keys.map(lambda value: test_set_check(value, 0.2))
assert coordinate_membership.groupby(coordinate_keys).nunique().max() == 1

check_partition(housing, strat_train_set, strat_test_set)
assert housing.income_cat.max() == 5 and housing.income_cat.min() == 1
full = housing.income_cat.value_counts(normalize=True).sort_index()
sample = strat_test_set.income_cat.value_counts(normalize=True).sort_index()
max_error = (full - sample).abs().max()
assert max_error <= 1 / len(strat_test_set)

print("All partition, reproducibility, stable-ID and stratification checks passed.")
print("Coordinate-ID duplicate rows:", duplicate_rows)
print(f"Maximum income-category proportion error: {max_error:.8f}")


## Sources

- [Original code repository](https://github.com/youssefHosni/Machine-Learning-Practical-Guide)
- [Housing dataset from Hands-On Machine Learning, second edition](https://github.com/ageron/handson-ml2/tree/master/datasets/housing)
- [StratifiedShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html)
